# QuantumCircuit.jl Phase 3A Single-Mode Drive Detuning and Coupled Response


## Outline

This notebook starts with the simplest useful detuning example: a single two-level mode driven on and off resonance.
It then closes with a short coupled-mode extension so you can see how the same drive intuition carries into transfer through a coupling.

1. Build single-mode intuition for resonant versus detuned drive.
2. Visualize how detuning suppresses excited-state population.
3. Extend the same idea to a short coupled-response example.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using UnicodePlots


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Compare resonant and detuned drive on a two-level mode

A two-level resonator is enough to see the main effect.
When the drive frequency matches the mode frequency, the excited-state population grows efficiently.
If the drive is detuned, the response stays much smaller.


## Step 1A - Hamiltonian used in this notebook

The first half of this notebook uses a single driven resonator mode. In the current code path, that means

$$
\hat H_{\mathrm{single}}(t) = \omega_r \, \hat a^\dagger \hat a + \Omega \cos(\omega_d t) \left( \hat a + \hat a^\dagger \right).
$$

The detuning question is simply whether the drive frequency $\omega_d$ is close to the local mode frequency $\omega_r$.
When $\omega_d \approx \omega_r$, the excited-state population grows efficiently; when the drive is detuned, the same drive amplitude produces a much weaker response.

The second half of the notebook extends that same idea to two coupled modes,

$$
\hat H_{\mathrm{pair}}(t) = \omega_{r1} \, \hat a_1^\dagger \hat a_1 + \omega_{r2} \, \hat a_2^\dagger \hat a_2 + g \left( \hat a_1^\dagger \hat a_2 + \hat a_1 \hat a_2^\dagger \right) + \Omega \cos(\omega_d t) \left( \hat a_1 + \hat a_1^\dagger \right).
$$

Only `r1` is driven directly, but the coupling term can move response into `r2`, which is why the later transfer plots are physically meaningful.


In [3]:
resonator = Resonator(:r1; ω = 1.0, dim = 2)
sys = CompositeSystem(resonator)
ψ0 = basis_state(sys; r1 = 0)
tlist = collect(range(0.0, 10.0; length = 201))

drive = SubsystemDrive(
    :r1_x_drive,
    :r1,
    :x,
    (p, t) -> p.Ω * cos(p.ωd * t),
)

function run_single_mode_drive(sys, ψ0, tlist, drive, ωd)
    return evolve(
        sys,
        ψ0,
        tlist;
        drives = [drive],
        observables = [ObservableSpec(:nr, :r1, :n), ObservableSpec(:xr, :r1, :x)],
        params = (; Ω = 0.35, ωd),
    )
end

on_resonance = run_single_mode_drive(sys, ψ0, tlist, drive, 1.0)
off_resonance = run_single_mode_drive(sys, ψ0, tlist, drive, 1.35)

on_excited = population_trace(on_resonance, :r1, 1)
off_excited = population_trace(off_resonance, :r1, 1)

(; max_on_resonance = maximum(on_excited.values), max_off_resonance = maximum(off_excited.values))


(max_on_resonance = 0.9935956042245369, max_off_resonance = 0.5181093211568131)

## Step 2 - Visualize the detuning effect

The resonance case should reach a clearly larger excited-state population than the detuned case.


In [4]:
detuning_plot = lineplot(
    on_excited.times,
    on_excited.values;
    name = "ωd = 1.0",
    title = "Excited-state population: resonance vs detuning",
    xlabel = "time",
    ylabel = "P(|1>)",
)
lineplot!(detuning_plot, off_excited.times, off_excited.values; name = "ωd = 1.35")
detuning_plot


            Excited-state population: resonance vs detuning          
            ┌────────────────────────────────────────┐          
          1 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⡤⠤⠤⠤⠴⠒⠋⠉⠉⠒⢄│ ωd = 1.0 
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ ωd = 1.35
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⣀⣀⡤⠖⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
   P(|1>)   │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠋⠁⠀⠀⠀⣠⠖⠑⠢⡀⠀⠀⠀⠀⠀⠀⣀⡀⠀⠀⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠁⠀⠀⠀⠀⡰⠃⠀⠀⠀⠙⢆⡀⠀⢀⡴⠋⠁⠙⢆⠀⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠃⠀⠀⠀⠀⡰⠁⠀⠀⠀⠀⠀⠀⠉⠚⠁⠀⠀⠀⠀⠀⢧⠀⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠗⠢⢤⣀⣀⠞⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢧⠀⠀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣔⠇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠣⡀⠀│          
            │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣾⠏⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉│          
            │⠀⠀⠀⢀⡤⠴⠒

## Step 3 - Look at population flow on resonance

For a two-level truncation, the ground-state and excited-state populations should move in opposite directions and approximately sum to one.


In [5]:
on_ground = population_trace(on_resonance, :r1, 0)

population_plot = lineplot(
    on_ground.times,
    on_ground.values;
    name = "P(|0>)",
    title = "Population flow under a resonant drive",
    xlabel = "time",
    ylabel = "population",
)
lineplot!(population_plot, on_excited.times, on_excited.values; name = "P(|1>)")
population_plot


                ⠀⠀Population flow under a resonant drive⠀⠀       
                ┌────────────────────────────────────────┐       
              1 │⠉⠑⠦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⡤⠤⠤⠤⠴⠒⠋⠉⠉⠒⢄│ P(|0>)
                │⠀⠀⠀⠈⠓⠲⠤⠖⠲⢤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ P(|1>)
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠱⡄⠀⠀⠀⠀⠀⠀⠀⣠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠱⡀⠀⣀⣀⣀⡤⠖⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
   population   │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡱⣏⡁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠁⠀⠉⠉⠉⠓⠦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠃⠀⠀⠀⠀⠀⠀⠀⠙⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠏⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
          

## Step 4 - Drive a coupled system and watch the response spread

Now use two resonators with a capacitive coupling.
Only `r1` is driven directly, but `r2` responds through the coupling term.


In [6]:
r1 = Resonator(:r1; ω = 1.0, dim = 2)
r2 = Resonator(:r2; ω = 1.0, dim = 2)
coupled_sys = CompositeSystem(
    r1,
    r2,
    CapacitiveCoupling(:r1, :r2; g = 0.08),
)

ψ0_coupled = basis_state(coupled_sys; r1 = 0, r2 = 0)
tlist_coupled = collect(range(0.0, 16.0; length = 321))

coupled_drive = SubsystemDrive(
    :r1_x_drive,
    :r1,
    :x,
    (p, t) -> p.Ω * cos(p.ωd * t),
)

coupled_result = evolve(
    coupled_sys,
    ψ0_coupled,
    tlist_coupled;
    drives = [coupled_drive],
    observables = [ObservableSpec(:n1, :r1, :n), ObservableSpec(:n2, :r2, :n)],
    params = (; Ω = 0.25, ωd = 1.0),
)

r1_excited = population_trace(coupled_result, :r1, 1)
r2_excited = population_trace(coupled_result, :r2, 1)

(; max_r1_excited = maximum(r1_excited.values), max_r2_excited = maximum(r2_excited.values))


(max_r1_excited = 0.7998003435372326, max_r2_excited = 0.5015616257085669)

## Step 5 - Plot local and transferred response

The `r1` trace is the directly driven response.
The `r2` trace shows how much excitation is transferred through the coupling.


In [7]:
transfer_plot = lineplot(
    r1_excited.times,
    r1_excited.values;
    name = "r1 excited",
    title = "Driven response in a coupled two-mode system",
    xlabel = "time",
    ylabel = "P(|1>)",
)
lineplot!(transfer_plot, r2_excited.times, r2_excited.values; name = "r2 excited")
transfer_plot


              Driven response in a coupled two-mode system           
              ┌────────────────────────────────────────┐           
          0.8 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠉⠙⢦⣀⡴⠋⠉⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ r1 excited
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⠃⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⠤⠤⠀⠀⠀⠀⠀⠀⠀⠀│ r2 excited
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠏⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡴⠲⠤⠞⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠁⠀⠀⠀⠀⠀⠀⠀⠀│           
   P(|1>)     │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡜⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⢀⡔⠒⠚⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⠀⡜⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⠀⢰⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
              │⠀⠀⠀⠀⠀⢀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡤⠚⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

## Step 6 - Compare resonant and detuned transfer in the coupled system

Detuning the same drive reduces not only the local excitation but also the response transferred to the coupled partner.


In [8]:
coupled_detuned = evolve(
    coupled_sys,
    ψ0_coupled,
    tlist_coupled;
    drives = [coupled_drive],
    observables = [ObservableSpec(:n1, :r1, :n), ObservableSpec(:n2, :r2, :n)],
    params = (; Ω = 0.25, ωd = 1.25),
)

detuned_r2_excited = population_trace(coupled_detuned, :r2, 1)

comparison_plot = lineplot(
    r2_excited.times,
    r2_excited.values;
    name = "ωd = 1.0",
    title = "Transferred response: resonance vs detuning",
    xlabel = "time",
    ylabel = "P_r2(|1>)",
)
lineplot!(comparison_plot, detuned_r2_excited.times, detuned_r2_excited.values; name = "ωd = 1.25")
comparison_plot


                 Transferred response: resonance vs detuning          
                 ┌────────────────────────────────────────┐          
             0.6 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ ωd = 1.0 
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ ωd = 1.25
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡜⠁⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡜⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
   P_r2(|1>)     │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡜⠁⠀⠀⠀⠀⠀⣀⣠⠤⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠋⠀⠀⢀⡠⠴⠚⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠁⢀⡤⠞⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│          
                 │⠀

## Interpretation

- `population_trace(result, :r1, 1)` is often more intuitive than `observable_trace(..., :nr)` when the local Hilbert space is only two levels.
- In the single-mode example, resonance produces a much stronger population swing than detuning.
- In the coupled example, the second mode responds even though it is not driven directly.
- Detuning suppresses both the local response and the transferred response.


## Current Limits

- These examples are closed-system only. There is no relaxation or dephasing in Phase 3A.
- The drive API is still subsystem-local. Coupling drives and pulse scheduling belong to later phases.
- Two-level examples are useful for intuition, but larger truncations are still needed for realistic weakly anharmonic devices.


## Exercise

Increase the coupling `g` or the drive amplitude `Ω` and rerun the coupled example.
Watch how the `r2` response changes relative to the directly driven `r1` response.


In [ ]:
stronger_coupled_sys = CompositeSystem(
    Resonator(:r1; ω = 1.0, dim = 2),
    Resonator(:r2; ω = 1.0, dim = 2),
    CapacitiveCoupling(:r1, :r2; g = 0.14),
)

stronger_result = evolve(
    stronger_coupled_sys,
    basis_state(stronger_coupled_sys; r1 = 0, r2 = 0),
    tlist_coupled;
    drives = [SubsystemDrive(:r1_x_drive, :r1, :x, (p, t) -> p.Ω * cos(p.ωd * t))],
    observables = [ObservableSpec(:n1, :r1, :n), ObservableSpec(:n2, :r2, :n)],
    params = (; Ω = 0.25, ωd = 1.0),
)

stronger_r2 = population_trace(stronger_result, :r2, 1)
exercise_plot = lineplot(
    r2_excited.times,
    r2_excited.values;
    name = "g = 0.08",
    title = "Exercise: stronger coupling increases transferred response",
    xlabel = "time",
    ylabel = "P_r2(|1>)",
)
lineplot!(exercise_plot, stronger_r2.times, stronger_r2.values; name = "g = 0.14")
exercise_plot
